In [0]:
%pip install hdbcli

In [0]:
from hdbcli import dbapi

def conectar(usuario, chave_senha):
    return dbapi.connect(
        address=dbutils.secrets.get("s4lake", "hana_host"),
        port=443,
        user=usuario,
        password=dbutils.secrets.get("s4lake", chave_senha),
        encrypt=True,
        sslValidateCertificate=True,
    )

SCHEMA = "SAPHANADB"

def c(n):
    return f"NVARCHAR({n})"

DATA = "NVARCHAR(8)"

ITENS_TITULO = [
    ("MANDT", c(3)), ("BUKRS", c(4)), ("KUNNR", c(10)), ("GJAHR", c(4)), ("BELNR", c(10)),
    ("BUZEI", c(3)), ("BUDAT", DATA), ("BLDAT", DATA), ("BLART", c(2)), ("SHKZG", c(1)),
    ("WAERS", c(5)), ("DMBTR", "DECIMAL(13,2)"), ("WRBTR", "DECIMAL(13,2)"), ("ZFBDT", DATA),
    ("ZTERM", c(4)), ("ZBD1T", "DECIMAL(3,0)"), ("VBELN", c(10)), ("AUGDT", DATA), ("AUGBL", c(10)),
]

TABELAS = {
    "KNA1": [("MANDT", c(3)), ("KUNNR", c(10)), ("LAND1", c(3)), ("NAME1", c(35)), ("ORT01", c(35)),
             ("REGIO", c(3)), ("STCD1", c(16)), ("KTOKD", c(4)), ("BRSCH", c(4)), ("ERDAT", DATA)],
    "KNB1": [("MANDT", c(3)), ("KUNNR", c(10)), ("BUKRS", c(4)), ("AKONT", c(10)), ("ZTERM", c(4)),
             ("ERDAT", DATA)],
    "VBAK": [("MANDT", c(3)), ("VBELN", c(10)), ("ERDAT", DATA), ("AUART", c(4)), ("VKORG", c(4)),
             ("VTWEG", c(2)), ("SPART", c(2)), ("KUNNR", c(10)), ("NETWR", "DECIMAL(15,2)"),
             ("WAERK", c(5)), ("BUKRS_VF", c(4)), ("ZTERM", c(4))],
    "VBAP": [("MANDT", c(3)), ("VBELN", c(10)), ("POSNR", c(6)), ("MATNR", c(40)), ("ARKTX", c(40)),
             ("KWMENG", "DECIMAL(15,3)"), ("VRKME", c(3)), ("NETWR", "DECIMAL(15,2)"), ("WAERK", c(5)),
             ("ERDAT", DATA)],
    "VBRK": [("MANDT", c(3)), ("VBELN", c(10)), ("FKART", c(4)), ("FKDAT", DATA), ("KUNAG", c(10)),
             ("BUKRS", c(4)), ("NETWR", "DECIMAL(15,2)"), ("MWSBK", "DECIMAL(13,2)"), ("WAERK", c(5)),
             ("ZTERM", c(4))],
    "VBRP": [("MANDT", c(3)), ("VBELN", c(10)), ("POSNR", c(6)), ("AUBEL", c(10)), ("AUPOS", c(6)),
             ("MATNR", c(40)), ("FKIMG", "DECIMAL(13,3)"), ("VRKME", c(3)), ("NETWR", "DECIMAL(15,2)")],
    "BSID": ITENS_TITULO,
    "BSAD": ITENS_TITULO,
}

conn = conectar("S4LAKE_CARGA", "hana_senha_carga")
cur = conn.cursor()

for tabela, colunas in TABELAS.items():
    try:
        cur.execute(f'DROP TABLE "{SCHEMA}"."{tabela}"')
    except dbapi.Error:
        pass
    definicao = ", ".join(f'"{nome}" {tipo}' for nome, tipo in colunas)
    cur.execute(f'CREATE COLUMN TABLE "{SCHEMA}"."{tabela}" ({definicao})')
    print(f"{tabela}: criada com {len(colunas)} colunas")

conn.close()

In [0]:
import pandas as pd
from decimal import Decimal

LANDING = "/Volumes/workspace/s4lake_bronze/landing"
LOTE = 5000

def converter(valor, eh_decimal, limite):
    if eh_decimal:
        return Decimal(valor) if valor != "" else None
    if limite is not None and len(valor) > limite:
        return valor[:limite]
    return valor

conn = conectar("S4LAKE_CARGA", "hana_senha_carga")
conn.setautocommit(False)
cur = conn.cursor()

for tabela, colunas in TABELAS.items():
    nomes = [nome for nome, _ in colunas]
    decimais = [tipo.startswith("DECIMAL") for _, tipo in colunas]
    limites = [
        int(re.match(r"NVARCHAR\((\d+)\)", tipo).group(1)) if tipo.startswith("NVARCHAR") else None
        for _, tipo in colunas
    ]

    df = pd.read_csv(f"{LANDING}/{tabela}.csv", dtype=str, keep_default_na=False)[nomes]

    for nome, limite in zip(nomes, limites):
        if limite is not None:
            cortados = (df[nome].str.len() > limite).sum()
            if cortados > 0:
                print(f"{tabela}.{nome}: {cortados} valores truncados para {limite} caracteres")

    linhas = [
        tuple(converter(v, d, l) for v, d, l in zip(registro, decimais, limites))
        for registro in df.itertuples(index=False, name=None)
    ]

    marcadores = ", ".join("?" for _ in nomes)
    sql = f'INSERT INTO "{SCHEMA}"."{tabela}" VALUES ({marcadores})'

    cur.execute(f'TRUNCATE TABLE "{SCHEMA}"."{tabela}"')
    for i in range(0, len(linhas), LOTE):
        cur.executemany(sql, linhas[i:i + LOTE])
    conn.commit()

    cur.execute(f'SELECT COUNT(*) FROM "{SCHEMA}"."{tabela}"')
    print(f"{tabela:5} CSV: {len(df):>7,}  HANA: {cur.fetchone()[0]:>7,}")

conn.close()

In [0]:
import re

for tabela, colunas in TABELAS.items():
    df = pd.read_csv(f"{LANDING}/{tabela}.csv", dtype=str, keep_default_na=False)
    for nome, tipo in colunas:
        m = re.match(r"NVARCHAR\((\d+)\)", tipo)
        if m:
            limite = int(m.group(1))
            maior = df[nome].str.len().max()
            acima = (df[nome].str.len() > limite).sum()
            if acima > 0:
                print(f"{tabela}.{nome}: limite {limite}, maior {maior}, {acima} valores acima")